# 13 — GRC: Governance, Risk & Compliance Frameworks

**Author:** Bency (Benjamin Adjei)  
**Course:** M.S. Cybersecurity — Cybersecurity Governance & Risk Management  
**Certifications:** CISA, CISM, CompTIA CySA+

---

## Objectives
- Understand the GRC triad and its role in enterprise security
- Map key compliance frameworks (NIST CSF, ISO 27001, SOC 2, HIPAA, PCI-DSS)
- Perform a qualitative risk assessment
- Build a risk register with scoring and treatment plans
- Conduct a compliance gap analysis
- Generate an executive-ready GRC report

## 1. Setup & Imports

In [ ]:
import json
from datetime import datetime, timezone
from collections import Counter

## 2. The GRC Triad

```
        ┌──────────────┐
        │  GOVERNANCE  │  — Policies, accountability, strategic direction
        └──────┬───────┘
               │
    ┌──────────┴──────────┐
    │                     │
┌───▼──────┐       ┌──────▼──────┐
│   RISK   │       │ COMPLIANCE  │
│ Identify │       │ Frameworks  │
│ Assess   │       │ Audits      │
│ Treat    │       │ Controls    │
└──────────┘       └─────────────┘
```

### Major Compliance Frameworks

| Framework | Scope | Key Focus |
|-----------|-------|-----------|
| **NIST CSF 2.0** | All industries | Identify, Protect, Detect, Respond, Recover, Govern |
| **ISO 27001:2022** | All industries | ISMS — 93 controls across 4 themes |
| **SOC 2 Type II** | SaaS / cloud | Trust Service Criteria — Security, Availability, Confidentiality |
| **HIPAA** | Healthcare | PHI protection — administrative, physical, technical safeguards |
| **PCI-DSS v4.0** | Payment cards | 12 requirements protecting cardholder data |
| **GDPR** | EU data subjects | Data privacy, consent, breach notification (72 hrs) |
| **CMMC 2.0** | US DoD contractors | 110 NIST SP 800-171 practices across 3 levels |

## 3. Risk Assessment Methodology

**Risk Score = Likelihood × Impact**

| Score | Likelihood | Impact |
|-------|-----------|--------|
| 1 | Rare (< 1/year) | Negligible |
| 2 | Unlikely (1–2/year) | Minor |
| 3 | Possible (quarterly) | Moderate |
| 4 | Likely (monthly) | Major |
| 5 | Almost certain (weekly+) | Catastrophic |

In [ ]:
def score_risk(likelihood: int, impact: int) -> dict:
    """Score a risk and return rating, colour, and treatment priority."""
    score = likelihood * impact
    if score >= 15:
        rating, treatment = 'CRITICAL', 'Immediate action — escalate to CISO'
    elif score >= 10:
        rating, treatment = 'HIGH',     'Address within 30 days'
    elif score >= 5:
        rating, treatment = 'MEDIUM',   'Address within 90 days'
    else:
        rating, treatment = 'LOW',      'Accept or monitor annually'
    return {'score': score, 'rating': rating, 'treatment_guidance': treatment}


# Risk heat-map preview
print('=== RISK HEAT MAP (Likelihood × Impact) ===\n')
print(f'{"":12}', end='')
for imp in range(1, 6):
    print(f'Impact={imp}  ', end='')
print()
for lik in range(5, 0, -1):
    print(f'Likely={lik}   ', end='')
    for imp in range(1, 6):
        r = score_risk(lik, imp)
        cell = f"{r['score']:>2}({r['rating'][0]})  "
        print(cell, end='')
    print()

## 4. Risk Register

In [ ]:
RISK_REGISTER = [
    {
        'id'        : 'RSK-001',
        'category'  : 'Access Control',
        'risk'      : 'Privileged accounts lack MFA — attacker could compromise admin credentials',
        'likelihood': 4,
        'impact'    : 5,
        'owner'     : 'IT Security',
        'treatment' : 'Mitigate',
        'controls'  : ['Enforce MFA via Azure AD Conditional Access', 'Privileged Identity Management (PIM)']
    },
    {
        'id'        : 'RSK-002',
        'category'  : 'Data Protection',
        'risk'      : 'PII data stored unencrypted in legacy database — breach could trigger GDPR fine',
        'likelihood': 3,
        'impact'    : 5,
        'owner'     : 'Data Engineering',
        'treatment' : 'Mitigate',
        'controls'  : ['Encrypt database at rest with AES-256', 'Implement column-level encryption for PII']
    },
    {
        'id'        : 'RSK-003',
        'category'  : 'Third-Party Risk',
        'risk'      : 'Critical SaaS vendor has no SOC 2 report — supply chain compromise risk',
        'likelihood': 2,
        'impact'    : 4,
        'owner'     : 'Procurement / GRC',
        'treatment' : 'Transfer',
        'controls'  : ['Require SOC 2 Type II in vendor contracts', 'Annual third-party security questionnaire']
    },
    {
        'id'        : 'RSK-004',
        'category'  : 'Vulnerability Management',
        'risk'      : 'Internet-facing systems unpatched > 90 days — known exploitable CVEs present',
        'likelihood': 4,
        'impact'    : 4,
        'owner'     : 'IT Operations',
        'treatment' : 'Mitigate',
        'controls'  : ['Monthly patching cycle', 'Critical CVEs patched within 7 days (SLA)']
    },
    {
        'id'        : 'RSK-005',
        'category'  : 'Business Continuity',
        'risk'      : 'No tested disaster recovery plan — ransomware could cause extended outage',
        'likelihood': 2,
        'impact'    : 5,
        'owner'     : 'CISO / IT Leadership',
        'treatment' : 'Mitigate',
        'controls'  : ['Develop and test DR plan annually', 'Immutable offsite backups (3-2-1 rule)']
    },
    {
        'id'        : 'RSK-006',
        'category'  : 'Insider Threat',
        'risk'      : 'Terminated employees retain system access beyond offboarding — data theft risk',
        'likelihood': 3,
        'impact'    : 3,
        'owner'     : 'HR / IT Security',
        'treatment' : 'Mitigate',
        'controls'  : ['Automated deprovisioning on HR termination event', 'Access review quarterly']
    },
    {
        'id'        : 'RSK-007',
        'category'  : 'Physical Security',
        'risk'      : 'Server room lacks biometric access control — tailgating risk',
        'likelihood': 2,
        'impact'    : 3,
        'owner'     : 'Facilities',
        'treatment' : 'Mitigate',
        'controls'  : ['Install badge + PIN access', 'CCTV with 90-day retention']
    },
    {
        'id'        : 'RSK-008',
        'category'  : 'Compliance',
        'risk'      : 'No formal security awareness training programme — staff susceptible to phishing',
        'likelihood': 5,
        'impact'    : 3,
        'owner'     : 'HR / Security',
        'treatment' : 'Mitigate',
        'controls'  : ['Annual mandatory security awareness training', 'Monthly phishing simulations']
    }
]

# Score all risks
for risk in RISK_REGISTER:
    scored = score_risk(risk['likelihood'], risk['impact'])
    risk.update(scored)

# Print register sorted by score
print('=== RISK REGISTER ===\n')
print(f'{"ID":<9} {"Score":<7} {"Rating":<10} {"Category":<25} Risk')
print('-' * 100)
for r in sorted(RISK_REGISTER, key=lambda x: x['score'], reverse=True):
    print(f"{r['id']:<9} {r['score']:<7} {r['rating']:<10} {r['category']:<25} {r['risk'][:65]}")

## 5. NIST CSF 2.0 Compliance Gap Analysis

In [ ]:
# Simulated organisation maturity against NIST CSF 2.0 functions
# Tier 1=Partial, 2=Risk Informed, 3=Repeatable, 4=Adaptive
NIST_CSF_ASSESSMENT = [
    {
        'function'   : 'GV — Govern',
        'description': 'Cybersecurity risk management strategy, roles, policies',
        'target_tier': 3,
        'current_tier': 2,
        'gaps'       : ['No formal cybersecurity strategy document', 'Roles & responsibilities not fully defined'],
        'actions'    : ['Develop cybersecurity strategy aligned to business objectives', 'Define RACI for security roles']
    },
    {
        'function'   : 'ID — Identify',
        'description': 'Asset management, risk assessment, supply chain risk',
        'target_tier': 3,
        'current_tier': 2,
        'gaps'       : ['Asset inventory incomplete', 'No formal third-party risk programme'],
        'actions'    : ['Deploy asset discovery tool (e.g. Qualys, Tenable)', 'Implement vendor risk management process']
    },
    {
        'function'   : 'PR — Protect',
        'description': 'Access control, awareness training, data security, secure config',
        'target_tier': 3,
        'current_tier': 1,
        'gaps'       : ['MFA not enforced universally', 'No security awareness training programme', 'Inconsistent patch management'],
        'actions'    : ['Enforce MFA via Conditional Access', 'Launch phishing simulation programme', 'Automate patching with WSUS/Intune']
    },
    {
        'function'   : 'DE — Detect',
        'description': 'Continuous monitoring, anomaly detection, log management',
        'target_tier': 3,
        'current_tier': 2,
        'gaps'       : ['SIEM not fully tuned — high false positive rate', 'EDR not deployed on all endpoints'],
        'actions'    : ['Tune SIEM correlation rules', 'Deploy EDR to 100% of endpoints']
    },
    {
        'function'   : 'RS — Respond',
        'description': 'Incident response plan, communications, analysis, mitigation',
        'target_tier': 3,
        'current_tier': 2,
        'gaps'       : ['IR plan exists but not tested in 2+ years', 'No tabletop exercises conducted'],
        'actions'    : ['Conduct annual IR tabletop exercise', 'Update IR playbooks for ransomware and phishing']
    },
    {
        'function'   : 'RC — Recover',
        'description': 'Recovery planning, improvements, communications',
        'target_tier': 3,
        'current_tier': 1,
        'gaps'       : ['No DR plan tested', 'Backup integrity not verified', 'No recovery time objectives (RTO/RPO) defined'],
        'actions'    : ['Define and document RTO/RPO for critical systems', 'Test backup restoration quarterly']
    }
]

print('=== NIST CSF 2.0 GAP ANALYSIS ===\n')
print(f'{"Function":<20} {"Current":>8} {"Target":>7} {"Gap":>5}  Status')
print('-' * 65)
for fn in NIST_CSF_ASSESSMENT:
    gap    = fn['target_tier'] - fn['current_tier']
    status = 'AT TARGET' if gap == 0 else f'{gap} tier(s) below target'
    bar    = '█' * fn['current_tier'] + '░' * (4 - fn['current_tier'])
    print(f"{fn['function']:<20} Tier {fn['current_tier']}/4 [{bar}]  Gap={gap}  {status}")
    for g in fn['gaps']:
        print(f'    ⚠ {g}')

total_gap = sum(f['target_tier'] - f['current_tier'] for f in NIST_CSF_ASSESSMENT)
print(f'\nTotal maturity gap: {total_gap} tier-points across {len(NIST_CSF_ASSESSMENT)} functions')

## 6. Control Mapping Across Frameworks

In [ ]:
# How a single control maps across multiple frameworks
CONTROL_CROSSWALK = [
    {
        'control'   : 'Multi-Factor Authentication (MFA)',
        'NIST_CSF'  : 'PR.AA-01',
        'ISO_27001' : 'A.8.5',
        'SOC2'      : 'CC6.1',
        'PCI_DSS'   : 'Req 8.4',
        'HIPAA'     : '§164.312(d)',
        'status'    : 'PARTIAL'
    },
    {
        'control'   : 'Encryption at Rest',
        'NIST_CSF'  : 'PR.DS-01',
        'ISO_27001' : 'A.8.24',
        'SOC2'      : 'CC9.1',
        'PCI_DSS'   : 'Req 3.5',
        'HIPAA'     : '§164.312(a)(2)(iv)',
        'status'    : 'PARTIAL'
    },
    {
        'control'   : 'Security Awareness Training',
        'NIST_CSF'  : 'PR.AT-01',
        'ISO_27001' : 'A.6.3',
        'SOC2'      : 'CC1.4',
        'PCI_DSS'   : 'Req 12.6',
        'HIPAA'     : '§164.308(a)(5)',
        'status'    : 'NOT IMPLEMENTED'
    },
    {
        'control'   : 'Incident Response Plan',
        'NIST_CSF'  : 'RS.MA-01',
        'ISO_27001' : 'A.5.26',
        'SOC2'      : 'CC7.3',
        'PCI_DSS'   : 'Req 12.10',
        'HIPAA'     : '§164.308(a)(6)',
        'status'    : 'IMPLEMENTED'
    },
    {
        'control'   : 'Vulnerability Management',
        'NIST_CSF'  : 'ID.RA-01',
        'ISO_27001' : 'A.8.8',
        'SOC2'      : 'CC7.1',
        'PCI_DSS'   : 'Req 6.3',
        'HIPAA'     : '§164.308(a)(1)',
        'status'    : 'PARTIAL'
    },
    {
        'control'   : 'Access Reviews',
        'NIST_CSF'  : 'PR.AA-05',
        'ISO_27001' : 'A.8.2',
        'SOC2'      : 'CC6.2',
        'PCI_DSS'   : 'Req 7.2',
        'HIPAA'     : '§164.308(a)(3)',
        'status'    : 'NOT IMPLEMENTED'
    }
]

STATUS_ICON = {'IMPLEMENTED': '✅', 'PARTIAL': '⚠️ ', 'NOT IMPLEMENTED': '❌'}

print('=== CONTROL CROSSWALK ===\n')
print(f'{"Control":<30} {"NIST CSF":<12} {"ISO 27001":<11} {"SOC 2":<8} {"PCI-DSS":<10} Status')
print('-' * 95)
for c in CONTROL_CROSSWALK:
    icon = STATUS_ICON.get(c['status'], '?')
    print(f"{c['control']:<30} {c['NIST_CSF']:<12} {c['ISO_27001']:<11} {c['SOC2']:<8} {c['PCI_DSS']:<10} {icon} {c['status']}")

status_counts = Counter(c['status'] for c in CONTROL_CROSSWALK)
print(f'\nSummary: {dict(status_counts)}')

## 7. Executive GRC Report

In [ ]:
risk_by_rating = Counter(r['rating'] for r in RISK_REGISTER)
csf_avg_tier   = sum(f['current_tier'] for f in NIST_CSF_ASSESSMENT) / len(NIST_CSF_ASSESSMENT)

grc_report = {
    'report_generated' : datetime.now(timezone.utc).isoformat(),
    'organisation'     : 'Example Corp',
    'prepared_by'      : 'Bency (Benjamin Adjei)',
    'reporting_period' : 'Q1 2025',
    'executive_summary': {
        'overall_risk_posture' : 'HIGH',
        'critical_risks'       : risk_by_rating.get('CRITICAL', 0),
        'high_risks'           : risk_by_rating.get('HIGH', 0),
        'nist_csf_avg_tier'    : round(csf_avg_tier, 1),
        'nist_csf_target_tier' : 3,
        'controls_implemented' : status_counts.get('IMPLEMENTED', 0),
        'controls_partial'     : status_counts.get('PARTIAL', 0),
        'controls_missing'     : status_counts.get('NOT IMPLEMENTED', 0)
    },
    'top_risks': [
        {k: v for k, v in r.items() if k not in ('treatment_guidance',)}
        for r in sorted(RISK_REGISTER, key=lambda x: x['score'], reverse=True)[:5]
    ],
    'nist_csf_gaps': [
        {'function': f['function'], 'gap': f['target_tier'] - f['current_tier'],
         'priority_actions': f['actions']}
        for f in NIST_CSF_ASSESSMENT if f['current_tier'] < f['target_tier']
    ],
    'strategic_recommendations': [
        'Enforce MFA across all privileged and internet-facing accounts (30 days)',
        'Encrypt all databases containing PII — GDPR breach risk is material (60 days)',
        'Launch mandatory security awareness training and phishing simulation (90 days)',
        'Conduct DR tabletop exercise and validate backup restoration (90 days)',
        'Implement quarterly access reviews and automated deprovisioning (60 days)',
        'Engage external auditor for ISO 27001 gap assessment (next quarter)'
    ]
}

print(json.dumps(grc_report, indent=2))

## 8. Key Takeaways

| Concept | Takeaway |
|---------|----------|
| **Risk = Likelihood × Impact** | Quantify to prioritise — gut feeling is not a risk methodology |
| **Risk treatment** | Mitigate, Transfer (insurance/contract), Accept, or Avoid |
| **Framework selection** | Pick the framework that fits your industry and regulatory exposure |
| **Control crosswalk** | One control can satisfy multiple frameworks — avoid redundant effort |
| **NIST CSF tiers** | Tier 3 (Repeatable) is the realistic enterprise target — not perfection |
| **GRC is ongoing** | Not a one-time audit — review the risk register quarterly |

### GRC Tools (Reference)
| Tool | Use Case |
|------|----------|
| **ServiceNow GRC** | Enterprise risk & compliance management |
| **OneTrust** | Privacy, risk, and compliance automation |
| **Archer** | Risk management and audit management |
| **Vanta / Drata** | SOC 2 / ISO 27001 automation for SaaS |
| **OSCAL** | NIST machine-readable security controls format |

---
*Series Complete — You have built a full Cybersecurity Lab Notebook series (Notebooks 06–13)*